# Library vs custom codebook vector quantization on Vision Transformer

This notebook compares scikit-learn KMeans with the project’s custom mathematical NumPy codebook vector quantization on the existing fine-tuned ViT-B/16 CIFAR-10 classifier. Both implementations use identical vector grouping, codebook size, tensor selection, storage precision, preprocessing, and evaluation data. No fake-quantization API is used.

The compressed representation consists of integer codebook assignments plus stored centroids. For prediction, weights are explicitly reconstructed and evaluated with FP32 Keras operations and FP32 activations; this measures weight reconstruction accuracy and storage, not compressed inference speed.


In [1]:
from math import ceil, log2
from pathlib import Path
import gc
import json
import sys
import time

import keras
import keras_hub
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.cluster import KMeans
from sklearn.metrics import pairwise_distances_argmin

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models import CIFAR10_CLASS_NAMES
from src.models.transformer_based import (
    VIT_BASE_PATCH16_224_IMAGENET_PRESET,
    build_vit_image_preprocessor,
)
from src.quantization.custom_quantization import (
    CodebookQuantizedTensor,
    ModelCodebookQuantizationResult,
    codebook_vectorize_model,
    estimate_gradient_importance,
    quantization_mse,
    reconstruct_codebook_model_weights,
    vectorize_array,
)

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"scikit-learn: {__import__('sklearn').__version__}")


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TensorFlow: 2.20.0
Keras: 3.14.1
scikit-learn: 1.9.0


## 1. Configuration

This accuracy-oriented profile uses a 256-entry scalar codebook: one 8-bit assignment represents one weight. The patch projection, class/position embeddings, biases, normalization tensors, and final classifier remain FP32. The custom path uses bounded gradient-sensitivity refinement; scikit-learn remains the unweighted 8-bit baseline.


In [2]:
PRESET = VIT_BASE_PATCH16_224_IMAGENET_PRESET
IMAGE_SIZE = (224, 224)
CLASS_NAMES = CIFAR10_CLASS_NAMES

MODEL_PATH = (
    PROJECT_ROOT / "artifacts" / "vit_cifar10_ptq"
    / "vit_cifar10_fp32.keras"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "vit_codebook_vq_comparison"

VECTOR_DIM = 1
CODEBOOK_SIZE = 256
CODEBOOK_DTYPE = np.float32
KMEANS_ITERATIONS = 30
MAX_KMEANS_SAMPLES = 100_000
KMEANS_TOLERANCE = 1e-6
NUM_EVALUATION_SAMPLES = None  # None evaluates all 10,000 CIFAR-10 test images.
EVALUATION_BATCH_SIZE = 16
RANDOM_SEED = 42
PRESERVE_PATCH_EMBEDDING = True
PRESERVE_FINAL_CLASSIFIER = True
NUM_IMPORTANCE_CALIBRATION_SAMPLES = 32
IMPORTANCE_BATCH_SIZE = 4

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing fine-tuned classifier: {MODEL_PATH}. Run notebook 10 first."
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSIGNMENT_BITS = max(1, ceil(log2(CODEBOOK_SIZE)))
print(f"Assignment bits per vector: {ASSIGNMENT_BITS}")
print(f"Assignment bits per scalar: {ASSIGNMENT_BITS / VECTOR_DIM:.2f}")


Assignment bits per vector: 8
Assignment bits per scalar: 8.00


## 2. Load CIFAR-10 and matching ViT preprocessing

The experiment uses the same CIFAR-10 test set and ViT-B/16 preprocessor as Notebook 10. Preprocessing remains outside the codebook-quantized model.


In [3]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.cifar10.load_data()
train_labels = train_labels.reshape(-1).astype(np.int32)
test_labels = test_labels.reshape(-1).astype(np.int32)
if NUM_EVALUATION_SAMPLES is not None:
    test_images = test_images[:NUM_EVALUATION_SAMPLES]
    test_labels = test_labels[:NUM_EVALUATION_SAMPLES]

preprocessor = build_vit_image_preprocessor()

def preprocess_images(images):
    return preprocessor(images)

print(f"Evaluation images: {len(test_images):,}")
print(f"Raw image shape: {test_images.shape}")
print(f"Preprocessed shape: {preprocess_images(test_images[:1]).shape}")


Evaluation images: 10,000
Raw image shape: (10000, 32, 32, 3)
Preprocessed shape: (1, 224, 224, 3)


## 3. Load existing ViT weights and define tensor selection

Only internal transformer `kernel` tensors are codebook candidates. Rank-2 attention biases are correctly preserved as biases. The patch projection, class token, positional embedding, and final predictions kernel remain FP32 in this accuracy-oriented profile.


In [4]:
model = keras.models.load_model(MODEL_PATH, compile=False)

def tensor_name(weight):
    return getattr(weight, "path", weight.name)

eligible_names = [
    tensor_name(weight)
    for weight in model.weights
    if np.issubdtype(weight.numpy().dtype, np.floating)
    and weight.numpy().ndim >= 2
]
codebook_candidate_names = {
    tensor_name(weight)
    for weight in model.weights
    if np.issubdtype(weight.numpy().dtype, np.floating)
    and weight.numpy().ndim >= 2
    and tensor_name(weight).endswith("/kernel")
}
if PRESERVE_PATCH_EMBEDDING:
    codebook_candidate_names = {
        name for name in codebook_candidate_names
        if not name.endswith("patch_embedding/kernel")
    }
if PRESERVE_FINAL_CLASSIFIER:
    codebook_candidate_names = {
        name for name in codebook_candidate_names
        if not name.endswith("predictions/kernel")
    }

PRESERVED_TENSOR_NAMES = set(eligible_names) - codebook_candidate_names
print(f"Model parameter tensors: {len(model.weights)}")
print(f"Codebook-quantized tensors: {len(codebook_candidate_names)}")
print(f"Preserved rank >= 2 tensors: {len(PRESERVED_TENSOR_NAMES)}")
print("Explicitly accuracy-sensitive preserved tensors:")
for name in sorted(PRESERVED_TENSOR_NAMES):
    if ("embedding" in name or "class_token" in name
            or name.endswith("predictions/kernel")):
        print(f"  - {name}")
model.summary(expand_nested=True)


Model parameter tensors: 200
Codebook-quantized tensors: 72
Preserved rank >= 2 tensors: 40
Explicitly accuracy-sensitive preserved tensors:
  - predictions/kernel
  - vit_patching_and_embedding/class_token
  - vit_patching_and_embedding/patch_embedding/kernel
  - vit_patching_and_embedding/position_embedding/embeddings


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "vit_cifar10_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                  ┃ Output Shape                       ┃             Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━┩
│ images (InputLayer)                           │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ vi_t_backbone (ViTBackbone)                   │ (None, 197, 768)                   │          85,798,656 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ images (InputLayer)                      │ (None, 224, 224, 3)                │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ vit_patching_and_embedding               │ (None, 197, 768)                   │             742,656 │
│ (ViTPatchingAndEmbedding)                     │                                    │                     │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│    └ vit_encoder (ViTEncoder)                 │ (None, 197, 768)                   │          85,056,000 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ get_item (GetItem)                            │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ output_dropout (Dropout)                      │ (None, 768)                        │                   0 │
├───────────────────────────────────────────────┼────────────────────────────────────┼─────────────────────┤
│ predictions (Dense)                           │ (None, 10)                         │               7,690 │
└───────────────────────────────────────────────┴────────────────────────────────────┴─────────────────────┘

 Total params: 85,806,346 (327.33 MB)

 Trainable params: 85,806,346 (327.33 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Inbuilt/library codebook vectorization with scikit-learn

Scikit-learn performs k-means++ initialization, Lloyd updates, and nearest-centroid assignment. The wrapper only converts Keras tensors to sub-vectors and records the compressed representation.


In [5]:
def assignment_dtype(codebook_size):
    if codebook_size <= 256:
        return np.uint8
    if codebook_size <= 65536:
        return np.uint16
    return np.uint32

def sklearn_vector_quantize_array(
    array, *, name, vector_dim, codebook_size, codebook_dtype,
    max_samples, iterations, tolerance, seed
):
    vectors, layout = vectorize_array(array, vector_dim, axis=-1)
    flat_vectors = vectors.reshape(-1, vector_dim)
    rng = np.random.default_rng(seed)
    if max_samples is not None and len(flat_vectors) > max_samples:
        fit_vectors = flat_vectors[rng.choice(
            len(flat_vectors), size=max_samples, replace=False
        )]
    else:
        fit_vectors = flat_vectors

    estimator = KMeans(
        n_clusters=codebook_size,
        init="k-means++",
        n_init=1,
        max_iter=iterations,
        tol=tolerance,
        random_state=seed,
        algorithm="lloyd",
    )
    estimator.fit(fit_vectors)
    stored_codebook = estimator.cluster_centers_.astype(codebook_dtype)
    assignments = pairwise_distances_argmin(
        flat_vectors, stored_codebook, metric="euclidean"
    ).reshape(vectors.shape[:2]).astype(assignment_dtype(codebook_size))
    return CodebookQuantizedTensor(
        name=name,
        assignments=assignments,
        codebooks=stored_codebook,
        layout=layout,
        mode="vector",
        original_dtype=str(np.asarray(array).dtype),
        assignment_bits=max(1, ceil(log2(codebook_size))),
    )

def sklearn_codebook_vectorize_model(keras_model):
    tensors = []
    skipped = []
    original_bytes = 0
    compressed_bytes = 0
    for tensor_index, weight in enumerate(keras_model.weights):
        values = weight.numpy()
        name = tensor_name(weight)
        original_bytes += values.nbytes
        if name not in codebook_candidate_names:
            skipped.append(name)
            compressed_bytes += values.nbytes
            continue
        result = sklearn_vector_quantize_array(
            values,
            name=name,
            vector_dim=VECTOR_DIM,
            codebook_size=CODEBOOK_SIZE,
            codebook_dtype=CODEBOOK_DTYPE,
            max_samples=MAX_KMEANS_SAMPLES,
            iterations=KMEANS_ITERATIONS,
            tolerance=KMEANS_TOLERANCE,
            seed=RANDOM_SEED + tensor_index,
        )
        tensors.append(result)
        compressed_bytes += result.estimated_compressed_size_bytes
    return ModelCodebookQuantizationResult(
        tensors=tensors,
        skipped_tensor_names=skipped,
        original_size_bytes=original_bytes,
        estimated_compressed_size_bytes=compressed_bytes,
    )


In [6]:
def sensitivity_batches():
    count = min(NUM_IMPORTANCE_CALIBRATION_SAMPLES, len(train_images))
    for start_index in range(0, count, IMPORTANCE_BATCH_SIZE):
        stop = min(start_index + IMPORTANCE_BATCH_SIZE, count)
        yield preprocess_images(train_images[start_index:stop]), train_labels[start_index:stop]

gradient_importance = estimate_gradient_importance(
    model, sensitivity_batches(),
    included_tensor_names=codebook_candidate_names, use_square_root=True
)
start = time.perf_counter()
sklearn_result = sklearn_codebook_vectorize_model(model)
sklearn_quantization_seconds = time.perf_counter() - start

sklearn_model = keras.models.load_model(MODEL_PATH, compile=False)
sklearn_model.set_weights(
    reconstruct_codebook_model_weights(model, sklearn_result)
)
print(f"scikit-learn codebook construction: {sklearn_quantization_seconds:.2f} s")
print(f"Quantized tensors: {len(sklearn_result.tensors)}")


/Users/sahil/Group Studies/Quantization-Group-Studies/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


scikit-learn codebook construction: 63.28 s
Quantized tensors: 72


## 5. Custom mathematical codebook vectorization

This path uses the project's NumPy k-means++ and explicit nearest-codeword mathematics with bounded calibration-gradient sensitivity. It uses the same true-W8 scalar representation and tensor selection as the library baseline.


In [7]:
start = time.perf_counter()
custom_result = codebook_vectorize_model(
    model,
    vector_dim=VECTOR_DIM,
    codebook_size=CODEBOOK_SIZE,
    mode="vector",
    quantize_min_rank=2,
    excluded_tensor_names={
        tensor_name(weight) for weight in model.weights
        if tensor_name(weight) not in codebook_candidate_names
    },
    importance_by_name=gradient_importance,
    codebook_dtype=CODEBOOK_DTYPE,
    max_kmeans_samples=MAX_KMEANS_SAMPLES,
    kmeans_iterations=KMEANS_ITERATIONS,
    seed=RANDOM_SEED,
)
custom_quantization_seconds = time.perf_counter() - start

custom_model = keras.models.load_model(MODEL_PATH, compile=False)
custom_model.set_weights(
    reconstruct_codebook_model_weights(model, custom_result)
)
gc.collect()
print(f"Custom codebook construction: {custom_quantization_seconds:.2f} s")
print(f"Quantized tensors: {len(custom_result.tensors)}")


Custom codebook construction: 257.87 s
Quantized tensors: 72


## 6. Tensor coverage, reconstruction error, and memory


In [8]:
original_by_name = {tensor_name(weight): weight.numpy() for weight in model.weights}
sklearn_by_name = {tensor.name: tensor for tensor in sklearn_result.tensors}
custom_by_name = {tensor.name: tensor for tensor in custom_result.tensors}
assert sklearn_by_name.keys() == custom_by_name.keys() == codebook_candidate_names

comparison_rows = []
for name, sklearn_tensor in sklearn_by_name.items():
    custom_tensor = custom_by_name[name]
    original = original_by_name[name]
    comparison_rows.append({
        "tensor": name,
        "rank": original.ndim,
        "shape": tuple(original.shape),
        "values": original.size,
        "sklearn_mse": quantization_mse(original, sklearn_tensor),
        "custom_mse": quantization_mse(original, custom_tensor),
        "sklearn_compressed_bytes": sklearn_tensor.estimated_compressed_size_bytes,
        "custom_compressed_bytes": custom_tensor.estimated_compressed_size_bytes,
    })
layer_comparison = pd.DataFrame(comparison_rows).sort_values(
    "custom_mse", ascending=False
)

original_bytes = sklearn_result.original_size_bytes
original_values = int(sum(weight.numpy().size for weight in model.weights))
quantized_values = int(sum(original_by_name[name].size for name in sklearn_by_name))
comparison_summary = pd.DataFrame([
    {"model": "Original FP32 ViT", "implementation": "none", "quantized_tensors": 0, "quantized_values": 0, "parameter_bytes": original_bytes, "quantization_seconds": 0.0},
    {"model": "scikit-learn codebook VQ", "implementation": "sklearn.cluster.KMeans", "quantized_tensors": len(sklearn_result.tensors), "quantized_values": quantized_values, "parameter_bytes": sklearn_result.estimated_compressed_size_bytes, "quantization_seconds": sklearn_quantization_seconds},
    {"model": "Custom codebook VQ", "implementation": "NumPy k-means", "quantized_tensors": len(custom_result.tensors), "quantized_values": quantized_values, "parameter_bytes": custom_result.estimated_compressed_size_bytes, "quantization_seconds": custom_quantization_seconds},
])
comparison_summary["parameter_mib"] = comparison_summary["parameter_bytes"] / 1024**2
comparison_summary["compression_ratio_vs_fp32"] = original_bytes / comparison_summary["parameter_bytes"]
comparison_summary["memory_reduction_percent"] = (
    1 - comparison_summary["parameter_bytes"] / original_bytes
) * 100
comparison_summary["effective_bits_per_all_parameter_value"] = (
    comparison_summary["parameter_bytes"] * 8 / original_values
)

display(comparison_summary)
display(layer_comparison.head(15))


,model,implementation,quantized_tensors,quantized_values,parameter_bytes,quantization_seconds,parameter_mib,compression_ratio_vs_fp32,memory_reduction_percent,effective_bits_per_all_parameter_value
0,Original FP32 ViT,none,0,0,343225384,0.000000,327.325233,1.000000,0.000000,32.000000
1,scikit-learn codebook VQ,sklearn.cluster.KMeans,72,84934656,88495144,63.284706,84.395546,3.878466,74.216609,8.250685
2,Custom codebook VQ,NumPy k-means,72,84934656,88495144,257.868744,84.395546,3.878466,74.216609,8.250685


,tensor,rank,shape,values,sklearn_mse,custom_mse,sklearn_compressed_bytes,custom_compressed_bytes
35,vit_encoder/transformer_block_6/mlp/dense_2/ke...,2,"(3072, 768)",2359296,0.000049,0.000049,2360320,2360320
40,vit_encoder/transformer_block_7/mlp/dense_1/ke...,2,"(768, 3072)",2359296,0.000007,0.000007,2360320,2360320
47,vit_encoder/transformer_block_8/mlp/dense_2/ke...,2,"(3072, 768)",2359296,0.000006,0.000006,2360320,2360320
41,vit_encoder/transformer_block_7/mlp/dense_2/ke...,2,"(3072, 768)",2359296,0.000005,0.000005,2360320,2360320
29,vit_encoder/transformer_block_5/mlp/dense_2/ke...,2,"(3072, 768)",2359296,0.000005,0.000005,2360320,2360320
53,vit_encoder/transformer_block_9/mlp/dense_2/ke...,2,"(3072, 768)",2359296,0.000005,0.000005,2360320,2360320
59,vit_encoder/transformer_block_10/mlp/dense_2/k...,2,"(3072, 768)",2359296,0.000004,0.000004,2360320,2360320
3,vit_encoder/transformer_block_1/mha/attention_...,3,"(12, 64, 768)",589824,0.000003,0.000003,590848,590848
1,vit_encoder/transformer_block_1/mha/key/kernel,3,"(768, 12, 64)",589824,0.000003,0.000003,590848,590848
71,vit_encoder/transformer_block_12/mlp/dense_2/k...,2,"(3072, 768)",2359296,0.000002,0.000002,2360320,2360320


### Parameter tensors grouped by rank and selection


In [9]:
rank_rows = []
for weight in model.weights:
    values = weight.numpy()
    name = tensor_name(weight)
    rank_rows.append({
        "tensor": name,
        "rank": values.ndim,
        "shape": tuple(values.shape),
        "values": values.size,
        "codebook_quantized": name in codebook_candidate_names,
        "preserved": name not in codebook_candidate_names,
        "role": (
            "embedding" if ("embedding" in name or "class_token" in name)
            else "kernel" if name.endswith("/kernel")
            else "bias_or_normalization"
        ),
    })
rank_details = pd.DataFrame(rank_rows)
rank_summary = rank_details.groupby("rank", as_index=False).agg(
    total_tensors=("tensor", "count"),
    total_values=("values", "sum"),
    quantized_tensors=("codebook_quantized", "sum"),
    preserved_tensors=("preserved", "sum"),
)
rank_summary


,rank,total_tensors,total_values,quantized_tensors,preserved_tensors
0,1,88,94474,0,88
1,2,62,56809728,24,38
2,3,49,28312320,48,1
3,4,1,589824,0,1


## 7. CIFAR-10 predictions and accuracy comparison

All rows use identical test images and matching ViT preprocessing. Full evaluation may take a long time on CPU; set `NUM_EVALUATION_SAMPLES` for a quick development run.


In [10]:
def predict_keras_labels(keras_model, raw_images, batch_size=EVALUATION_BATCH_SIZE):
    predictions = []
    for start in range(0, len(raw_images), batch_size):
        images = preprocess_images(raw_images[start:start + batch_size])
        logits = keras_model(images, training=False).numpy()
        predictions.append(np.argmax(logits, axis=1))
    return np.concatenate(predictions)

print("1/3 Evaluating original FP32 ViT...")
fp32_predictions = predict_keras_labels(model, test_images)
print("2/3 Evaluating scikit-learn codebook ViT...")
sklearn_predictions = predict_keras_labels(sklearn_model, test_images)
print("3/3 Evaluating custom codebook ViT...")
custom_predictions = predict_keras_labels(custom_model, test_images)

fp32_accuracy = float(np.mean(fp32_predictions == test_labels))
sklearn_accuracy = float(np.mean(sklearn_predictions == test_labels))
custom_accuracy = float(np.mean(custom_predictions == test_labels))
accuracy_table = pd.DataFrame([
    {"model": "Original FP32 ViT", "codebook": "none", "correct_predictions": int(np.sum(fp32_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": fp32_accuracy * 100, "drop_from_fp32_percentage_points": 0.0},
    {"model": "scikit-learn codebook VQ", "codebook": "true W8 scalar KMeans; input/output tensors FP32", "correct_predictions": int(np.sum(sklearn_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": sklearn_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - sklearn_accuracy) * 100},
    {"model": "Custom codebook VQ", "codebook": "true W8 sensitivity-refined NumPy k-means; input/output tensors FP32", "correct_predictions": int(np.sum(custom_predictions == test_labels)), "test_images": len(test_labels), "accuracy_percent": custom_accuracy * 100, "drop_from_fp32_percentage_points": (fp32_accuracy - custom_accuracy) * 100},
])
accuracy_table


1/3 Evaluating original FP32 ViT...
2/3 Evaluating scikit-learn codebook ViT...
3/3 Evaluating custom codebook ViT...


,model,codebook,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ViT,none,9573,10000,95.73,0.00
1,scikit-learn codebook VQ,true W8 scalar KMeans; input/output tensors FP32,9572,10000,95.72,0.01
2,Custom codebook VQ,true W8 sensitivity-refined NumPy k-means; inp...,9575,10000,95.75,-0.02


### One-image prediction check


In [11]:
sample_index = 0
sample_table = pd.DataFrame([
    {"model": "Original FP32", "prediction": CLASS_NAMES[fp32_predictions[sample_index]]},
    {"model": "scikit-learn codebook VQ", "prediction": CLASS_NAMES[sklearn_predictions[sample_index]]},
    {"model": "Custom codebook VQ", "prediction": CLASS_NAMES[custom_predictions[sample_index]]},
])
sample_table.insert(1, "true_class", CLASS_NAMES[test_labels[sample_index]])
sample_table["correct"] = sample_table["prediction"] == sample_table["true_class"]
sample_table


,model,true_class,prediction,correct
0,Original FP32,cat,cat,True
1,scikit-learn codebook VQ,cat,cat,True
2,Custom codebook VQ,cat,cat,True


## 8. Save results


In [12]:
accuracy_path = OUTPUT_DIR / "codebook_accuracy_comparison.csv"
summary_path = OUTPUT_DIR / "codebook_memory_timing_comparison.csv"
layer_path = OUTPUT_DIR / "codebook_layer_reconstruction_comparison.csv"
rank_path = OUTPUT_DIR / "parameter_rank_summary.csv"
tensor_path = OUTPUT_DIR / "parameter_tensor_details.csv"
results_path = OUTPUT_DIR / "results.json"
accuracy_table.to_csv(accuracy_path, index=False)
comparison_summary.to_csv(summary_path, index=False)
layer_comparison.to_csv(layer_path, index=False)
rank_summary.to_csv(rank_path, index=False)
rank_details.to_csv(tensor_path, index=False)
results_path.write_text(json.dumps({
    "model": str(MODEL_PATH),
    "dataset": "CIFAR-10 test",
    "preset": PRESET,
    "image_size": IMAGE_SIZE,
    "evaluation_images": int(len(test_labels)),
    "vector_dim": VECTOR_DIM,
    "codebook_size": CODEBOOK_SIZE,
    "assignment_bits_per_vector": ASSIGNMENT_BITS,
    "assignment_bits_per_scalar": ASSIGNMENT_BITS / VECTOR_DIM,
    "quantized_tensor_names": sorted(codebook_candidate_names),
    "preserved_tensor_names": sorted(set(original_by_name) - codebook_candidate_names),
    "fp32_accuracy_percent": fp32_accuracy * 100,
    "sklearn_codebook_accuracy_percent": sklearn_accuracy * 100,
    "custom_codebook_accuracy_percent": custom_accuracy * 100,
    "activations_quantized": False,
}, indent=2) + "\n", encoding="utf-8")
print(f"Saved results under: {OUTPUT_DIR}")


Saved results under: /Users/sahil/Group Studies/Quantization-Group-Studies/artifacts/vit_codebook_vq_comparison


## Final ViT codebook-vector-quantization accuracy comparison (%)


In [13]:
final_accuracy_table = accuracy_table.copy()
final_accuracy_table["accuracy_percent"] = final_accuracy_table["accuracy_percent"].map(
    lambda value: f"{value:.2f}%"
)
final_accuracy_table["drop_from_fp32_percentage_points"] = final_accuracy_table[
    "drop_from_fp32_percentage_points"
].map(lambda value: f"{value:.2f}")
final_accuracy_table


,model,codebook,correct_predictions,test_images,accuracy_percent,drop_from_fp32_percentage_points
0,Original FP32 ViT,none,9573,10000,95.73%,0.00
1,scikit-learn codebook VQ,true W8 scalar KMeans; input/output tensors FP32,9572,10000,95.72%,0.01
2,Custom codebook VQ,true W8 sensitivity-refined NumPy k-means; inp...,9575,10000,95.75%,-0.02
